In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pybedtools
from upsetplot import UpSet, from_memberships
import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn2, venn3_circles
from collections import Counter

In [2]:
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
PREPROCESSING = Path("../preprocessing").resolve()
HIMORNA = DATA / "himorna"
BIOMART_INITIAL = DATA_INITIAL / "biomart"
GENCODE_INITIAL = DATA_INITIAL / "gencode"
GENCODE = DATA / "gencode"
ID_HISTORY_CONVERTER_RES = DATA / "id_history_converter_res"

In [3]:
def count_genome_coverage(anno, genome="hg38"):
    assembles_lenght = {"hg19":3101788170, "hg38":3099441038}
    anno_bed_sort = pybedtools.BedTool.from_dataframe(anno).sort()
    genome_coverage = anno_bed_sort.genome_coverage(bga=True, genome=genome)
    genome_coverage = genome_coverage.to_dataframe()
    genome_coverage["segment_lenght"] = genome_coverage["end"] - genome_coverage["start"]
    res = genome_coverage[genome_coverage["name"] != 0]["segment_lenght"].sum()
    return round(res / assembles_lenght[genome] * 100, 2)

In [4]:
def gencode_gtf_processing(to_gtf_file):
    gtf_file_GENCODE = to_gtf_file
    anno_GENCODE = pd.read_csv(gtf_file_GENCODE, sep="\t", header=None, skiprows=5)
    anno_GENCODE["ens_gene_primari"] = anno_GENCODE[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', ''))
    anno_GENCODE["ens_gene"] = anno_GENCODE[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', '').split(".")[0])#Такой способ парсинга не учитывает гены на Y хромосоме в псевдоаутосомных регионах.
    anno_GENCODE["gene_name"] = anno_GENCODE[8].apply(lambda x: x.split("gene_name")[1].split(" ")[1].replace('"', '').replace(';', ''))
    anno_GENCODE["unique_genome_reg_name"] = anno_GENCODE["ens_gene"] + "-" + anno_GENCODE[2]
    gencode_hg38 = anno_GENCODE[[0, 3, 4, "unique_genome_reg_name", 5, 6, "ens_gene_primari", "ens_gene", "gene_name"]]
    mask = gencode_hg38["unique_genome_reg_name"].apply(lambda x: "gene" in x)
    gencode_genes_hg38 = gencode_hg38[mask]
    gencode_genes_hg38 = gencode_genes_hg38.rename(columns={0:"chrom", 3:"start", 4:"end", "unique_genome_reg_name":"name", 5:"score", 6:"strand"})
    gencode_genes_hg38 = gencode_genes_hg38.drop_duplicates()
    #gencode_genes_hg38["gene_ens--gene_name"] = gencode_genes_hg38["ens_gene"] + "--" + gencode_genes_hg38["gene_name"]
    return gencode_genes_hg38.reset_index(drop=True)

In [5]:
def create_bad_genes_history_df(to_txt):
    bad_genes_history = pd.read_csv(to_txt, skipinitialspace=True)
    bad_genes_history = bad_genes_history.rename(columns=lambda x: x.strip())
    mask = bad_genes_history["Old stable ID"].apply(lambda x: "ENSG" in x)
    bad_genes_history = bad_genes_history[mask].reset_index(drop=True)
    bad_genes_history["Old stable ID without v"] = bad_genes_history["Old stable ID"].apply(lambda x: x.split(".")[0])
    bad_genes_history["New stable ID without v"] = bad_genes_history["New stable ID"].apply(lambda x: x.split(".")[0])
    return bad_genes_history

In [6]:
def check_retired_conditions(df):
    # Находим все строки с <retired>
    df = df.reset_index(drop=True)
    retired_rows = df[df['New stable ID without v'] == '<retired>']
    if len(retired_rows) == 0:
        return False
    # Проверка 1: что <retired> только в одном экземпляре
    is_single_instance = len(retired_rows) == 1
    
    # Получаем индекс строки с <retired>
    retired_index = retired_rows.index[0]
    total_rows = len(df)
    
    # Проверка 2: что <retired> не на последней строке
    is_not_last = retired_index != total_rows - 1
    
    # Проверка 3: что <retired> не на предпоследней строке  
    is_not_second_last = retired_index != total_rows - 2
    if is_not_second_last == False:
        if df.iloc[retired_index, 2] == df.iloc[retired_index-1, 2]:
            is_not_second_last = True
    if is_single_instance & is_not_last & is_not_second_last:
        return True
    return False

In [7]:
#нужно модифицировать функцию для случаев, когда ген вырождается в несколько
def bad_history_analyzer(ensg_id, bad_genes_history):
    query_res = bad_genes_history.query(f"`Old stable ID without v` == '{ensg_id}'")
    query_res_length = len(query_res)
    if query_res_length == 0:
        new_ensg = "<retired>"
        return new_ensg
    if check_retired_conditions(df=query_res):
        new_ensg = "<retired>-to-many"
        return new_ensg
    else:
        new_ensg = query_res.iloc[-1, 5]
    return new_ensg

In [8]:
def jaccard_calculating(df1, df2):
    df1_sortbed = pybedtools.BedTool.from_dataframe(df1[["chrom","start","end","gene_ens--gene_name","score","strand"]]).sort()
    df2_sortbed = pybedtools.BedTool.from_dataframe(df2[["chrom","start","end","gene_ens--gene_name","score","strand"]]).sort()
    df1_overlap_df2 = df1_sortbed.intersect(df2_sortbed, wao=True, s=True)
    df1_overlap_df2 = df1_overlap_df2.to_dataframe(disable_auto_names=True, header=None)
    df1_overlap_df2 = df1_overlap_df2.rename(columns={0:"f-chrom", 1:"f-strart", 2:"f-end", 3:"f-name", 4:"f-score", 5:"f-strand",
                                                6:"g-chrom", 7:"g-strart", 8:"g-end", 9:"g-name", 10:"g-score", 11:"g-strand",
                                                12:"overlap"})
    df1_overlap_df2["merge_len"] = df1_overlap_df2[["f-strart", "f-end", "g-strart", "g-end"]].max(axis=1) - df1_overlap_df2[["f-strart", "f-end", "g-strart", "g-end"]].min(axis=1)
    df1_overlap_df2["jaccard"] = np.round(df1_overlap_df2["overlap"] / df1_overlap_df2["merge_len"], 2)
    df1_overlap_df2["g-name"] = df1_overlap_df2["g-name"].replace(".", "--")
    df1_overlap_df2["gene_en-metric"] = \
                df1_overlap_df2["f-name"].apply(lambda x: x.split("--")[0]) == df1_overlap_df2["g-name"].apply(lambda x: x.split("--")[0])
    df1_overlap_df2["gene_name-metric"] = \
                df1_overlap_df2["f-name"].apply(lambda x: x.split("--")[1]) == df1_overlap_df2["g-name"].apply(lambda x: x.split("--")[1])
    df1_overlap_df2["f-ens_name"] = df1_overlap_df2["f-name"].apply(lambda x: x.split("--")[0])
    df1_overlap_df2["g-ens_name"] = df1_overlap_df2["g-name"].apply(lambda x: x.split("--")[0])
    return df1_overlap_df2

p.s. В старых аннотация gencode гены в псевдоаутосомных областях на X и Y хромосомах имели один gene_name, а ensg идентификатор отличался лишь постфиксом для Y. В новых версиях каждому геномному локусу соотвествует свой уникальный идентификатор ensg. В идеале, конечно, стоило бы в данном ноутбуке получать идентификаторы для этих генов отдельно через индекс Жаккара. Однако, сейчас я этого делать не буду, поскольку в himorna порезаны идентификаторы генов (они split(".")[0])

In [9]:
to_file = GENCODE_INITIAL / "gencode.v35.basic.annotation.gtf.gz"
gencode_v35_genes_hg38 = gencode_gtf_processing(to_gtf_file=to_file)
gencode_v35_genes_hg38.shape

(60656, 9)

In [10]:
gencode_v35_genes_hg38.head(1)

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1


In [11]:
gencode_v35_genes_hg38["ens_gene_primari"].apply(lambda x: x.split(".")[1]).unique()

array(['5', '1', '2', '3', '6', '7', '4', '10', '12', '11', '14', '9',
       '15', '17', '18', '21', '19', '20', '13', '8', '23', '16', '22',
       '29', '24', '27', '26', '25', '7_PAR_Y', '14_PAR_Y', '13_PAR_Y',
       '6_PAR_Y', '18_PAR_Y', '3_PAR_Y', '5_PAR_Y', '12_PAR_Y',
       '16_PAR_Y', '8_PAR_Y', '11_PAR_Y', '15_PAR_Y', '20_PAR_Y',
       '17_PAR_Y'], dtype=object)

In [12]:
gencode_v35_genes_hg38[gencode_v35_genes_hg38["ens_gene"] == "ENSG00000281849"]

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name
57634,chrX,386980,405579,ENSG00000281849-gene,.,+,ENSG00000281849.3,ENSG00000281849,AL732314.6
60058,chrY,386980,405579,ENSG00000281849-gene,.,+,ENSG00000281849.3_PAR_Y,ENSG00000281849,AL732314.6


In [13]:
to_file = GENCODE_INITIAL / "gencode.v47.basic.annotation.gtf.gz"
gencode_v47_genes_hg38 = gencode_gtf_processing(to_gtf_file=to_file)
gencode_v47_genes_hg38.shape

(78724, 9)

In [14]:
gencode_v47_genes_hg38.head(1)

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name
0,chr1,11121,24894,ENSG00000290825-gene,.,+,ENSG00000290825.2,ENSG00000290825,DDX11L16


In [15]:
gencode_v47_genes_hg38["ens_gene_primari"].apply(lambda x: x.split(".")[1]).unique()

array(['2', '6', '1', '3', '7', '8', '4', '11', '13', '5', '15', '10',
       '16', '18', '14', '12', '20', '21', '19', '22', '17', '9', '23',
       '24', '32', '25', '26', '29', '28', '27'], dtype=object)

In [17]:
mask = gencode_v35_genes_hg38["ens_gene_primari"].apply(lambda x: "PAR_Y" in x)
PAR_Y_indexes = gencode_v35_genes_hg38[mask].index
PAR_Y_indexes

Index([60053, 60054, 60055, 60056, 60057, 60058, 60059, 60060, 60061, 60062,
       60063, 60064, 60065, 60066, 60067, 60068, 60069, 60070, 60071, 60072,
       60073, 60074, 60075, 60076, 60077, 60078, 60079, 60080, 60081, 60082,
       60083, 60084, 60085, 60086, 60609, 60610, 60611, 60612, 60613, 60614,
       60615, 60616, 60617, 60618],
      dtype='int64')

In [18]:
len(PAR_Y_indexes)

44

In [19]:
for index in PAR_Y_indexes:
    gencode_v35_genes_hg38.loc[index, "ens_gene"] = gencode_v35_genes_hg38.loc[index, "ens_gene"] + "-PAR_Y"

In [20]:
mask = gencode_v35_genes_hg38["ens_gene_primari"].apply(lambda x: "PAR_Y" in x)
gencode_v35_genes_hg38[mask]

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name
60053,chrY,253743,255091,ENSG00000228572-gene,.,+,ENSG00000228572.7_PAR_Y,ENSG00000228572-PAR_Y,AL954722.1
60054,chrY,276322,303356,ENSG00000182378-gene,.,+,ENSG00000182378.14_PAR_Y,ENSG00000182378-PAR_Y,PLCXD1
60055,chrY,304529,318819,ENSG00000178605-gene,.,-,ENSG00000178605.13_PAR_Y,ENSG00000178605-PAR_Y,GTPBP6
60056,chrY,320990,321851,ENSG00000226179-gene,.,+,ENSG00000226179.6_PAR_Y,ENSG00000226179-PAR_Y,LINC00685
60057,chrY,333933,386955,ENSG00000167393-gene,.,-,ENSG00000167393.18_PAR_Y,ENSG00000167393-PAR_Y,PPP2R3B
60058,chrY,386980,405579,ENSG00000281849-gene,.,+,ENSG00000281849.3_PAR_Y,ENSG00000281849-PAR_Y,AL732314.6
60059,chrY,388100,388389,ENSG00000275287-gene,.,-,ENSG00000275287.5_PAR_Y,ENSG00000275287-PAR_Y,Metazoa_SRP
60060,chrY,419157,421980,ENSG00000280767-gene,.,+,ENSG00000280767.3_PAR_Y,ENSG00000280767-PAR_Y,AL732314.4
60061,chrY,523775,524102,ENSG00000234958-gene,.,-,ENSG00000234958.6_PAR_Y,ENSG00000234958-PAR_Y,FABP5P13
60062,chrY,545236,545352,ENSG00000229232-gene,.,-,ENSG00000229232.6_PAR_Y,ENSG00000229232-PAR_Y,KRT18P53


In [21]:
gencode_v35_genes_hg38["gene_ens--gene_name"] = gencode_v35_genes_hg38["ens_gene"] + "--" + gencode_v35_genes_hg38["gene_name"]
gencode_v47_genes_hg38["gene_ens--gene_name"] = gencode_v47_genes_hg38["ens_gene"] + "--" + gencode_v47_genes_hg38["gene_name"]

Существует несколько подходов к решению проблемы соотнесения двух аннотаций. Универсального варианта нет, поэтому мы будем использовать комбинированный подход:

Если ensg между разметками не меняется - считаем, что пара найдена. \
Если в новой версии разметки такого гена нет, то запускаем id_history_converter. \
Дополнительно, если по версии id_history_converter идентификатор данного гена устарел ("retired"), перепроверяем его путем пересечения геномных координат двух разметок, вводя порог на индекс Жаккара.

p.s. Подход коллег в https://pmc.ncbi.nlm.nih.gov/articles/PMC12322893/ работает только при отображении близких во времени аннотаций. Есть множество примеров, когда восстановить ensg вовсе невозможно по трем используемым в статье метрикам даже с гораздо более слабым порогом на индекс Жаккара. Контакт для связи: Аглетдинов Матвей, @Mathwey

In [22]:
mask = gencode_v35_genes_hg38["ens_gene"].isin(gencode_v47_genes_hg38["ens_gene"])
bad_genes = gencode_v35_genes_hg38[~mask]["ens_gene"].unique()
len(bad_genes)

1294

Именно эти 1294 идентификатор генов необходимо перевести на gencode47, если это возможно. Для этого воспользуемся id_history_converter. \
Он входит в набор программ ensembl-tools: https://github.com/Ensembl/ensembl-tools/tree/release/99 \
Для уставки: \
git clone git@github.com:Ensembl/ensembl-tools.git \
mamba create -n ensembl-tools -y \
mamba install perl-app-cpanminus -y \
cpanm Bio::EnsEMBL

Также можно/проще воспользоваться веб версией: \
https://www.ensembl.org/Homo_sapiens/Tools/IDMapper

In [23]:
with open(f"{ID_HISTORY_CONVERTER_RES}/bad-genes_id-gencode35_to_gencode47.txt", "w") as file:
    file.writelines("%s\n" % line for line in bad_genes)

In [24]:
to_txt = ID_HISTORY_CONVERTER_RES / "bad-genes_id-gencode35_to_gencode47-history.txt"
bad_genes_history = create_bad_genes_history_df(to_txt)
bad_genes_history

,Old stable ID,New stable ID,Release,Mapping score,Old stable ID without v,New stable ID without v
0,ENSG00000238009.1,ENSG00000238009.2,56,0.25645,ENSG00000238009,ENSG00000238009
1,ENSG00000238009.2,ENSG00000238009.3,76,0.973847,ENSG00000238009,ENSG00000238009
2,ENSG00000238009.3,ENSG00000238009.4,77,1,ENSG00000238009,ENSG00000238009
3,ENSG00000238009.4,ENSG00000238009.5,79,1,ENSG00000238009,ENSG00000238009
4,ENSG00000238009.5,ENSG00000238009.6,81,1,ENSG00000238009,ENSG00000238009
...,...,...,...,...,...,...
6026,ENSG00000226362.1,ENSG00000226362.2,71,0.614446,ENSG00000226362,ENSG00000226362
6027,ENSG00000226362.2,ENSG00000226362.2,98,0.9,ENSG00000226362,ENSG00000226362
6028,ENSG00000226362.2,ENSG00000226362.2,99,0.9,ENSG00000226362,ENSG00000226362
6029,ENSG00000226362.2,ENSG00000226362.2,100,0.9,ENSG00000226362,ENSG00000226362


In [25]:
bad_genes_history["Old stable ID without v"].nunique()

1246

In [26]:
len(bad_genes)

1294

По какой-то причне эти 4 идентификатора (помимо *-PAR_Y) не вошли в таблицу результатов. Полагаем, что они также устарели.

In [27]:
set(bad_genes) - set(bad_genes_history["Old stable ID without v"].unique())

{'ENSG00000002586-PAR_Y',
 'ENSG00000124333-PAR_Y',
 'ENSG00000124334-PAR_Y',
 'ENSG00000167393-PAR_Y',
 'ENSG00000169084-PAR_Y',
 'ENSG00000169093-PAR_Y',
 'ENSG00000169100-PAR_Y',
 'ENSG00000178605-PAR_Y',
 'ENSG00000182162-PAR_Y',
 'ENSG00000182378-PAR_Y',
 'ENSG00000182484-PAR_Y',
 'ENSG00000185203-PAR_Y',
 'ENSG00000185291-PAR_Y',
 'ENSG00000185960-PAR_Y',
 'ENSG00000196433-PAR_Y',
 'ENSG00000197976-PAR_Y',
 'ENSG00000198223-PAR_Y',
 'ENSG00000205755-PAR_Y',
 'ENSG00000214717-PAR_Y',
 'ENSG00000223274-PAR_Y',
 'ENSG00000223484-PAR_Y',
 'ENSG00000223511-PAR_Y',
 'ENSG00000223571-PAR_Y',
 'ENSG00000223773-PAR_Y',
 'ENSG00000225661-PAR_Y',
 'ENSG00000226179-PAR_Y',
 'ENSG00000227159-PAR_Y',
 'ENSG00000228410-PAR_Y',
 'ENSG00000228572-PAR_Y',
 'ENSG00000229232-PAR_Y',
 'ENSG00000230542-PAR_Y',
 'ENSG00000234622-PAR_Y',
 'ENSG00000234958-PAR_Y',
 'ENSG00000236017-PAR_Y',
 'ENSG00000236871-PAR_Y',
 'ENSG00000237040-PAR_Y',
 'ENSG00000237531-PAR_Y',
 'ENSG00000237801-PAR_Y',
 'ENSG000002

In [28]:
bad_genes_resolutions = {}
old_ids = bad_genes_history["Old stable ID without v"].unique()
for ensg_id in bad_genes:
    if ensg_id in old_ids:
        bad_genes_resolutions[ensg_id] = bad_history_analyzer(
            ensg_id=ensg_id, 
            bad_genes_history=bad_genes_history
        )
    else:
        bad_genes_resolutions[ensg_id] = "<retired>"

In [29]:
len(set(bad_genes))

1294

In [30]:
df_bad_genes_resolutions = pd.DataFrame({"name":bad_genes_resolutions.keys(), "name_latest":bad_genes_resolutions.values()})
df_bad_genes_resolutions

,name,name_latest
0,ENSG00000238009,ENSG00000241860
1,ENSG00000288531,ENSG00000228794
2,ENSG00000230699,ENSG00000272438
3,ENSG00000241180,ENSG00000272438
4,ENSG00000236948,ENSG00000289893
...,...,...
1289,ENSG00000124334-PAR_Y,<retired>
1290,ENSG00000270726-PAR_Y,<retired>
1291,ENSG00000185203-PAR_Y,<retired>
1292,ENSG00000182484-PAR_Y,<retired>


In [31]:
df_bad_genes_resolutions["name_latest2"] = np.where(df_bad_genes_resolutions["name"] == df_bad_genes_resolutions["name_latest"], "<retired>", df_bad_genes_resolutions["name_latest"])

In [32]:
df_bad_genes_resolutions

,name,name_latest,name_latest2
0,ENSG00000238009,ENSG00000241860,ENSG00000241860
1,ENSG00000288531,ENSG00000228794,ENSG00000228794
2,ENSG00000230699,ENSG00000272438,ENSG00000272438
3,ENSG00000241180,ENSG00000272438,ENSG00000272438
4,ENSG00000236948,ENSG00000289893,ENSG00000289893
...,...,...,...
1289,ENSG00000124334-PAR_Y,<retired>,<retired>
1290,ENSG00000270726-PAR_Y,<retired>,<retired>
1291,ENSG00000185203-PAR_Y,<retired>,<retired>
1292,ENSG00000182484-PAR_Y,<retired>,<retired>


In [33]:
mask = df_bad_genes_resolutions["name_latest2"].apply(lambda x: "ENSG" in x)
name_latest2_unique_df = pd.DataFrame(df_bad_genes_resolutions[mask]["name_latest2"].unique())
len(name_latest2_unique_df)

815

Проверим, все ли новые идентификаторы есть в gencode47

In [34]:
mask = name_latest2_unique_df[0].isin(gencode_v47_genes_hg38["ens_gene"])
name_latest2_unique_df[~mask]

,0
123,ENSG00000233143
403,ENSG00000225489


Восстановим актуальный идентификатор для этих генов руками:
ENSG00000233143 - ENSG00000223874
ENSG00000225489 - ENSG00000236924

In [35]:
assert "ENSG00000223874" in gencode_v47_genes_hg38["ens_gene"].to_list()
assert "ENSG00000236924" in gencode_v47_genes_hg38["ens_gene"].to_list()

In [36]:
df_bad_genes_resolutions[df_bad_genes_resolutions["name_latest2"] == "ENSG00000233143"]

,name,name_latest,name_latest2
174,ENSG00000236886,ENSG00000233143,ENSG00000233143
176,ENSG00000286601,ENSG00000233143,ENSG00000233143


In [37]:
df_bad_genes_resolutions[df_bad_genes_resolutions["name_latest2"] == "ENSG00000225489"]

,name,name_latest,name_latest2
606,ENSG00000229611,ENSG00000225489,ENSG00000225489


In [38]:
df_bad_genes_resolutions.iloc[174, 2] = "ENSG00000223874"
df_bad_genes_resolutions.iloc[176, 2] = "ENSG00000223874"
df_bad_genes_resolutions.iloc[606, 2] = "ENSG00000236924"

In [39]:
#Проверим, нет ли неотловленных состояний one-to-many (<retired>-to-many)
mask = df_bad_genes_resolutions["name"].duplicated()
assert mask.sum() == 0

In [40]:
mask = df_bad_genes_resolutions["name"] == df_bad_genes_resolutions["name_latest2"]
mask.sum()

0

In [38]:
df_bad_genes_resolutions

,name,name_latest,name_latest2
0,ENSG00000238009,ENSG00000241860,ENSG00000241860
1,ENSG00000288531,ENSG00000228794,ENSG00000228794
2,ENSG00000230699,ENSG00000272438,ENSG00000272438
3,ENSG00000241180,ENSG00000272438,ENSG00000272438
4,ENSG00000236948,ENSG00000289893,ENSG00000289893
...,...,...,...
1245,ENSG00000225393,ENSG00000224533,ENSG00000224533
1246,ENSG00000288621,<retired>,<retired>
1247,ENSG00000237563,ENSG00000147753,ENSG00000147753
1248,ENSG00000228890,ENSG00000147761,ENSG00000147761


In [41]:
mask = df_bad_genes_resolutions["name_latest2"].apply(lambda x: "retired" in x)
bad_genes_retired_gencode35 = df_bad_genes_resolutions[mask]["name"]
len(bad_genes_retired_gencode35)

328

Подсчет индекса Жаккара:

In [42]:
gencode_v35_overlap_gencode_v47 = jaccard_calculating(
    df1=gencode_v35_genes_hg38[["chrom","start","end","gene_ens--gene_name","score","strand"]],
    df2=gencode_v47_genes_hg38[["chrom","start","end","gene_ens--gene_name","score","strand"]]
)
gencode_v35_overlap_gencode_v47

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,g-score,g-strand,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name
0,chr1,11869,14409,ENSG00000223972--DDX11L1,.,+,chr1,12010,13670,ENSG00000223972--DDX11L1,.,+,1660,2540,0.65,True,True,ENSG00000223972,ENSG00000223972
1,chr1,11869,14409,ENSG00000223972--DDX11L1,.,+,chr1,11121,24894,ENSG00000290825--DDX11L16,.,+,2540,13773,0.18,False,False,ENSG00000223972,ENSG00000290825
2,chr1,14404,29570,ENSG00000227232--WASH7P,.,-,chr1,17369,17436,ENSG00000278267--MIR6859-1,.,-,67,15166,0.00,False,False,ENSG00000227232,ENSG00000278267
3,chr1,14404,29570,ENSG00000227232--WASH7P,.,-,chr1,14356,30744,ENSG00000310526--WASH7P,.,-,15166,16388,0.93,False,True,ENSG00000227232,ENSG00000310526
4,chr1,14404,29570,ENSG00000227232--WASH7P,.,-,chr1,14696,24886,ENSG00000227232--WASH7P,.,-,10190,15166,0.67,True,True,ENSG00000227232,ENSG00000227232
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92072,chrY,57201143,57203357,ENSG00000185203-PAR_Y--WASIR1,.,-,chrY,57201135,57203405,ENSG00000292370--WASIR1,.,-,2214,2270,0.98,False,True,ENSG00000185203-PAR_Y,ENSG00000292370
92073,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57207346,57212230,ENSG00000292372--WASH6P,.,+,4884,4884,1.00,False,True,ENSG00000182484-PAR_Y,ENSG00000292372
92074,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57190738,57208756,ENSG00000292369--ENSG00000292369,.,+,1410,21492,0.07,False,False,ENSG00000182484-PAR_Y,ENSG00000292369
92075,chrY,57212184,57214397,ENSG00000227159-PAR_Y--DDX11L16,.,-,chrY,57208516,57215634,ENSG00000310542--ENSG00000310542,.,-,2213,7118,0.31,False,False,ENSG00000227159-PAR_Y,ENSG00000310542


In [88]:
mask = gencode_v35_overlap_gencode_v47["f-ens_name"].apply(lambda x: "ENSG00000281849" in x)
gencode_v35_overlap_gencode_v47[mask]

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,g-score,g-strand,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name
87887,chrX,386980,405579,ENSG00000281849--AL732314.6,.,+,chrX,386962,511616,ENSG00000281849--ENSG00000281849,.,+,18599,124654,0.15,True,False,ENSG00000281849,ENSG00000281849
91299,chrY,386980,405579,ENSG00000281849-PAR_Y--AL732314.6,.,+,chrY,386962,511616,ENSG00000292349--ENSG00000292349,.,+,18599,124654,0.15,False,False,ENSG00000281849-PAR_Y,ENSG00000292349


Полностью полагаться на соответствие gene_name между сборками нельзя. По ним есть множество дублей:

In [45]:
gencode_v47_genes_hg38.groupby(by="gene_name").aggregate(
    Число_дублей=("ens_gene","nunique")
).query("Число_дублей >= 2").sort_values(by="Число_дублей", ascending=False)

,Число_дублей
gene_name,
Y_RNA,756
Metazoa_SRP,170
U3,50
U6,33
SNORA70,27
...,...
FRG1HP,2
FRG1BP,2
FOLH1B,2


Будем работать только с теми генами, которые по версии id_history_converter имеют статус "retired" \
Для них попробуем восстановить gene_ens, вводя порог на индекс Жакара >= 0.99

In [46]:
mask = df_bad_genes_resolutions["name_latest2"].apply(lambda x: x == "<retired>")
df_bad_genes_resolutions_retired = df_bad_genes_resolutions[mask]
df_bad_genes_resolutions_retired

,name,name_latest,name_latest2
5,ENSG00000277726,<retired>,<retired>
6,ENSG00000271895,<retired>,<retired>
10,ENSG00000234166,<retired>,<retired>
13,ENSG00000225986,<retired>,<retired>
22,ENSG00000281133,<retired>,<retired>
...,...,...,...
1289,ENSG00000124334-PAR_Y,<retired>,<retired>
1290,ENSG00000270726-PAR_Y,<retired>,<retired>
1291,ENSG00000185203-PAR_Y,<retired>,<retired>
1292,ENSG00000182484-PAR_Y,<retired>,<retired>


In [47]:
gencode_v35_overlap_gencode_v47_bad_genes_retired = pd.merge(left=gencode_v35_overlap_gencode_v47, right=df_bad_genes_resolutions_retired, how="inner", left_on="f-ens_name", right_on="name")
gencode_v35_overlap_gencode_v47_bad_genes_retired

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,...,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name,name,name_latest,name_latest2
0,chr1,11012662,11030528,ENSG00000277726--AL109811.3,.,+,chr1,11012344,11030528,ENSG00000120948--TARDBP,...,17866,18184,0.98,False,False,ENSG00000277726,ENSG00000120948,ENSG00000277726,<retired>,<retired>
1,chr1,11029659,11030528,ENSG00000271895--AL109811.2,.,+,chr1,11012344,11030528,ENSG00000120948--TARDBP,...,869,18184,0.05,False,False,ENSG00000271895,ENSG00000120948,ENSG00000271895,<retired>,<retired>
2,chr1,16197854,16198357,ENSG00000234166--ARHGEF19-AS1,.,+,.,-1,-1,--,...,0,16198358,0.00,False,False,ENSG00000234166,,ENSG00000234166,<retired>,<retired>
3,chr1,20184242,20186486,ENSG00000225986--UBXN10-AS1,.,-,chr1,20161253,20186518,ENSG00000187980--PLA2G2C,...,2244,25265,0.09,False,False,ENSG00000225986,ENSG00000187980,ENSG00000225986,<retired>,<retired>
4,chr1,45580892,45580996,ENSG00000281133--AL355480.3,.,-,.,-1,-1,--,...,0,45580997,0.00,False,False,ENSG00000281133,,ENSG00000281133,<retired>,<retired>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
455,chrY,57201143,57203357,ENSG00000185203-PAR_Y--WASIR1,.,-,chrY,57201135,57203405,ENSG00000292370--WASIR1,...,2214,2270,0.98,False,True,ENSG00000185203-PAR_Y,ENSG00000292370,ENSG00000185203-PAR_Y,<retired>,<retired>
456,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57207346,57212230,ENSG00000292372--WASH6P,...,4884,4884,1.00,False,True,ENSG00000182484-PAR_Y,ENSG00000292372,ENSG00000182484-PAR_Y,<retired>,<retired>
457,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57190738,57208756,ENSG00000292369--ENSG00000292369,...,1410,21492,0.07,False,False,ENSG00000182484-PAR_Y,ENSG00000292369,ENSG00000182484-PAR_Y,<retired>,<retired>
458,chrY,57212184,57214397,ENSG00000227159-PAR_Y--DDX11L16,.,-,chrY,57208516,57215634,ENSG00000310542--ENSG00000310542,...,2213,7118,0.31,False,False,ENSG00000227159-PAR_Y,ENSG00000310542,ENSG00000227159-PAR_Y,<retired>,<retired>


In [48]:
gencode_v35_overlap_gencode_v47_bad_genes_retired = gencode_v35_overlap_gencode_v47_bad_genes_retired.drop_duplicates()
len(gencode_v35_overlap_gencode_v47_bad_genes_retired)

460

In [49]:
mask = (gencode_v35_overlap_gencode_v47_bad_genes_retired["jaccard"] >= 0.99)
gencode_v35_overlap_gencode_v47_bad_genes_retired_jaccard_ok = gencode_v35_overlap_gencode_v47_bad_genes_retired[mask]
gencode_v35_overlap_gencode_v47_bad_genes_retired_jaccard_ok.head(3)

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,...,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name,name,name_latest,name_latest2
19,chr1,168695874,169103276,ENSG00000285622--AL135926.1,.,-,chr1,168695874,169103307,ENSG00000203601--LINC00970,...,407402,407433,1.0,False,False,ENSG00000285622,ENSG00000203601,ENSG00000285622,ENSG00000285622,<retired>
88,chr13,21287634,21287828,ENSG00000277077--AL158032.2,.,-,chr13,21287634,21287828,ENSG00000289133--ENSG00000289133,...,194,194,1.0,False,False,ENSG00000277077,ENSG00000289133,ENSG00000277077,<retired>,<retired>
97,chr14,26599755,27295516,ENSG00000258081--AL110292.1,.,+,chr14,26598369,27295516,ENSG00000257842--NOVA1-DT,...,695761,697147,1.0,False,False,ENSG00000258081,ENSG00000257842,ENSG00000258081,ENSG00000258081,<retired>


In [50]:
len(gencode_v35_overlap_gencode_v47_bad_genes_retired_jaccard_ok)

46

In [51]:
#Проверяем на множественные пересечения. Если падает с ошибкой, то нужно выбрать лучший вариант пересечения (возможно, тогда стоит дополнительно посмотреть на gene_name)
assert gencode_v35_overlap_gencode_v47_bad_genes_retired_jaccard_ok["f-name"].duplicated().sum() == 0

In [52]:
df_bad_genes_resolutions_jaccar_corrected = pd.merge(left=df_bad_genes_resolutions, right=gencode_v35_overlap_gencode_v47_bad_genes_retired_jaccard_ok[["f-ens_name","g-ens_name"]], how="left", left_on="name", right_on="f-ens_name")
df_bad_genes_resolutions_jaccar_corrected

,name,name_latest,name_latest2,f-ens_name,g-ens_name
0,ENSG00000238009,ENSG00000241860,ENSG00000241860,NaN,NaN
1,ENSG00000288531,ENSG00000228794,ENSG00000228794,NaN,NaN
2,ENSG00000230699,ENSG00000272438,ENSG00000272438,NaN,NaN
3,ENSG00000241180,ENSG00000272438,ENSG00000272438,NaN,NaN
4,ENSG00000236948,ENSG00000289893,ENSG00000289893,NaN,NaN
...,...,...,...,...,...
1289,ENSG00000124334-PAR_Y,<retired>,<retired>,NaN,NaN
1290,ENSG00000270726-PAR_Y,<retired>,<retired>,ENSG00000270726-PAR_Y,ENSG00000292369
1291,ENSG00000185203-PAR_Y,<retired>,<retired>,NaN,NaN
1292,ENSG00000182484-PAR_Y,<retired>,<retired>,ENSG00000182484-PAR_Y,ENSG00000292372


In [53]:
df_bad_genes_resolutions_jaccar_corrected["g-ens_name"] = df_bad_genes_resolutions_jaccar_corrected["g-ens_name"].fillna(df_bad_genes_resolutions_jaccar_corrected["name_latest2"])

In [54]:
mask = df_bad_genes_resolutions_jaccar_corrected["name_latest2"] == df_bad_genes_resolutions_jaccar_corrected["g-ens_name"]
len(df_bad_genes_resolutions_jaccar_corrected[~mask])

46

In [55]:
df_bad_genes_resolutions_jaccar_corrected = df_bad_genes_resolutions_jaccar_corrected.rename(columns={"g-ens_name":"name_latest3"})
df_bad_genes_resolutions_jaccar_corrected = df_bad_genes_resolutions_jaccar_corrected[["name", "name_latest2", "name_latest3"]]
df_bad_genes_resolutions_jaccar_corrected

,name,name_latest2,name_latest3
0,ENSG00000238009,ENSG00000241860,ENSG00000241860
1,ENSG00000288531,ENSG00000228794,ENSG00000228794
2,ENSG00000230699,ENSG00000272438,ENSG00000272438
3,ENSG00000241180,ENSG00000272438,ENSG00000272438
4,ENSG00000236948,ENSG00000289893,ENSG00000289893
...,...,...,...
1289,ENSG00000124334-PAR_Y,<retired>,<retired>
1290,ENSG00000270726-PAR_Y,<retired>,ENSG00000292369
1291,ENSG00000185203-PAR_Y,<retired>,<retired>
1292,ENSG00000182484-PAR_Y,<retired>,ENSG00000292372


In [85]:
df_bad_genes_resolutions_jaccar_corrected[df_bad_genes_resolutions_jaccar_corrected["name_latest3"] == "ENSG00000281849"]

,name,name_latest2,name_latest3
1221,ENSG00000280767,ENSG00000281849,ENSG00000281849


In [56]:
mask = df_bad_genes_resolutions_jaccar_corrected["name_latest2"].apply(lambda x: "ENSG" in x)
len(df_bad_genes_resolutions_jaccar_corrected[mask]["name"].unique())

966

In [57]:
mask = df_bad_genes_resolutions_jaccar_corrected["name_latest3"].apply(lambda x: "ENSG" in x)
len(df_bad_genes_resolutions_jaccar_corrected[mask]["name"].unique())

1012

Таким образом, в итоге нам удалось восстановить 1012 ens_gene.

In [58]:
premerged_1 = gencode_v35_genes_hg38.copy()
premerged_1["ens_gene-unchangeable"] = np.where(
    gencode_v35_genes_hg38["ens_gene"].isin(gencode_v47_genes_hg38["ens_gene"]),
    gencode_v35_genes_hg38["ens_gene"],
    np.nan
)
premerged_1

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name,gene_ens--gene_name,ens_gene-unchangeable
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1,ENSG00000223972
1,chr1,14404,29570,ENSG00000227232-gene,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232--WASH7P,ENSG00000227232
2,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267--MIR6859-1,ENSG00000278267
3,chr1,29554,31109,ENSG00000243485-gene,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485--MIR1302-2HG,ENSG00000243485
4,chr1,30366,30503,ENSG00000284332-gene,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332--MIR1302-2,ENSG00000284332
...,...,...,...,...,...,...,...,...,...,...,...
60651,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695--MT-ND6,ENSG00000198695
60652,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194--MT-TE,ENSG00000210194
60653,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727--MT-CYB,ENSG00000198727
60654,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195--MT-TT,ENSG00000210195


In [93]:
premerged_1["ens_gene-unchangeable"].isna().sum()

1294

In [60]:
premerged_2 = pd.merge(left=gencode_v35_genes_hg38, right=df_bad_genes_resolutions_jaccar_corrected[["name","name_latest3"]], how="left", left_on="ens_gene", right_on="name")
premerged_2

,chrom,start,end,name_x,score,strand,ens_gene_primari,ens_gene,gene_name,gene_ens--gene_name,name_y,name_latest3
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1,NaN,NaN
1,chr1,14404,29570,ENSG00000227232-gene,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232--WASH7P,NaN,NaN
2,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267--MIR6859-1,NaN,NaN
3,chr1,29554,31109,ENSG00000243485-gene,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485--MIR1302-2HG,NaN,NaN
4,chr1,30366,30503,ENSG00000284332-gene,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332--MIR1302-2,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
60651,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695--MT-ND6,NaN,NaN
60652,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194--MT-TE,NaN,NaN
60653,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727--MT-CYB,NaN,NaN
60654,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195--MT-TT,NaN,NaN


In [61]:
premerged_2["name_latest3"].isna().sum()

59362

In [94]:
59362 + 1294

60656

In [63]:
gencode_v35_to_v47_genes_hg38 = gencode_v35_genes_hg38.copy()
gencode_v35_to_v47_genes_hg38.head(1)

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name,gene_ens--gene_name
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1


In [64]:
gencode_v35_to_v47_genes_hg38["new-ens_gene"] = premerged_1["ens_gene-unchangeable"].fillna(premerged_2["name_latest3"])

In [65]:
assert gencode_v35_to_v47_genes_hg38["new-ens_gene"].isna().sum() == 0

In [66]:
del gencode_v35_to_v47_genes_hg38["name"]
del gencode_v35_to_v47_genes_hg38["gene_ens--gene_name"]

In [67]:
gencode_v35_to_v47_genes_hg38

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene
0,chr1,11869,14409,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972
1,chr1,14404,29570,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232
2,chr1,17369,17436,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267
3,chr1,29554,31109,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485
4,chr1,30366,30503,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332
...,...,...,...,...,...,...,...,...,...
60651,chrM,14149,14673,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695
60652,chrM,14674,14742,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194
60653,chrM,14747,15887,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727
60654,chrM,15888,15953,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195


In [95]:
mask = gencode_v35_to_v47_genes_hg38["ens_gene"] != gencode_v35_to_v47_genes_hg38["new-ens_gene"]
gencode_v35_to_v47_genes_hg38[mask]["ens_gene"].nunique()

1294

In [96]:
mask = (gencode_v35_to_v47_genes_hg38["ens_gene"] != gencode_v35_to_v47_genes_hg38["new-ens_gene"]) & (gencode_v35_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: "ENSG" in x))
gencode_v35_to_v47_genes_hg38[mask]["ens_gene"].nunique()

1012

In [97]:
mask = gencode_v35_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: "ENSG" not in x)
len(gencode_v35_to_v47_genes_hg38[mask])

282

In [98]:
mask = (gencode_v35_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: x == "<retired>"))
gencode_v35_to_v47_genes_hg38[mask]["ens_gene"].nunique()

271

In [99]:
mask = (gencode_v35_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: x == "<retired>-to-many"))
gencode_v35_to_v47_genes_hg38[mask]["ens_gene"].nunique()

11

Таким образом: \
1294 генов претерпели изменение \
Для 1012 из них удалось восстановить актуальный идентификатор в gencode47 \
282 устарели, найти соответствие для них не получилось. Причем 271 просто устарели, а 11 отобразились в несколько актуальных идентификаторов ("one-to-many")

Теперь добавим идентификаторы gene_name:

In [73]:
gencode_v35_to_v47_genes_hg38

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene
0,chr1,11869,14409,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972
1,chr1,14404,29570,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232
2,chr1,17369,17436,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267
3,chr1,29554,31109,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485
4,chr1,30366,30503,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332
...,...,...,...,...,...,...,...,...,...
60651,chrM,14149,14673,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695
60652,chrM,14674,14742,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194
60653,chrM,14747,15887,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727
60654,chrM,15888,15953,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195


In [74]:
gencode_v35_to_v47_genes_hg38_with_names = pd.merge(left=gencode_v35_to_v47_genes_hg38, right=gencode_v47_genes_hg38[["ens_gene","gene_name"]], how="left", left_on="new-ens_gene", right_on="ens_gene")
del gencode_v35_to_v47_genes_hg38_with_names["ens_gene_y"]
gencode_v35_to_v47_genes_hg38_with_names = gencode_v35_to_v47_genes_hg38_with_names.rename(columns={"ens_gene_x":"ens_gene","gene_name_x":"gene_name", "gene_name_y":"new-gene_name"})

In [75]:
mask = gencode_v35_to_v47_genes_hg38_with_names["new-gene_name"].isna()
gencode_v35_to_v47_genes_hg38_with_names[mask]

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name
374,chr1,11012662,11030528,.,+,ENSG00000277726.4,ENSG00000277726,AL109811.3,<retired>,NaN
376,chr1,11029659,11030528,.,+,ENSG00000271895.2,ENSG00000271895,AL109811.2,<retired>,NaN
548,chr1,16197854,16198357,.,+,ENSG00000234166.1,ENSG00000234166,ARHGEF19-AS1,<retired>,NaN
660,chr1,20184242,20186486,.,-,ENSG00000225986.1,ENSG00000225986,UBXN10-AS1,<retired>,NaN
1489,chr1,45580892,45580996,.,-,ENSG00000281133.1,ENSG00000281133,AL355480.3,<retired>,NaN
...,...,...,...,...,...,...,...,...,...,...
60073,chrY,1401769,1414028,.,+,ENSG00000236017.8_PAR_Y,ENSG00000236017-PAR_Y,ASMTL-AS1,<retired>,NaN
60078,chrY,1732584,1763212,.,+,ENSG00000223511.7_PAR_Y,ENSG00000223511-PAR_Y,AL683807.1,<retired>,NaN
60084,chrY,2609348,2657229,.,+,ENSG00000223773.7_PAR_Y,ENSG00000223773-PAR_Y,CD99P1,<retired>,NaN
60614,chrY,57184101,57197337,.,+,ENSG00000124334.17_PAR_Y,ENSG00000124334-PAR_Y,IL9R,<retired>,NaN


In [76]:
gencode_v35_to_v47_genes_hg38_with_names["gene_name"].isna().sum()

0

Заполним пустые ячейки в new-gene_name продублированными new-ens_gene. Точно также сделано в gencode47. Даже если для ensg гена нет gene_name, для него просто продублировано ens_gene.

In [77]:
gencode_v35_to_v47_genes_hg38_with_names["new-gene_name"] = gencode_v35_to_v47_genes_hg38_with_names["new-gene_name"].fillna(gencode_v35_to_v47_genes_hg38_with_names["new-ens_gene"])

In [78]:
gencode_v35_to_v47_genes_hg38_with_names

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name
0,chr1,11869,14409,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972,DDX11L1
1,chr1,14404,29570,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232,WASH7P
2,chr1,17369,17436,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267,MIR6859-1
3,chr1,29554,31109,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485,MIR1302-2HG
4,chr1,30366,30503,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332,MIR1302-2
...,...,...,...,...,...,...,...,...,...,...
60651,chrM,14149,14673,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695,MT-ND6
60652,chrM,14674,14742,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194,MT-TE
60653,chrM,14747,15887,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727,MT-CYB
60654,chrM,15888,15953,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195,MT-TT


In [79]:
gencode_v35_to_v47_genes_hg38_with_names["new-ens_gene"].isna().sum()

0

In [80]:
gencode_v35_to_v47_genes_hg38_with_names["new-gene_name"].isna().sum()

0

In [ ]:
chrX    HAVANA  gene    386962  511616  .       +       .       gene_id "ENSG00000281849.4"; gene_type "lncRNA"; gene_name "ENSG00000281849"; level 2; havana_gene "OTTHUMG00000189992.1";
chrY    HAVANA  gene    386962  511616  .       +       .       gene_id "ENSG00000292349.2"; gene_type "lncRNA"; gene_name "ENSG00000292349"; level 2; havana_gene "OTTHUMG00000189992.1";

In [86]:
mask = gencode_v35_to_v47_genes_hg38_with_names["ens_gene"].apply(lambda x: "ENSG00000281849" in x)
gencode_v35_to_v47_genes_hg38_with_names[mask]

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name
57634,chrX,386980,405579,.,+,ENSG00000281849.3,ENSG00000281849,AL732314.6,ENSG00000281849,ENSG00000281849
60058,chrY,386980,405579,.,+,ENSG00000281849.3_PAR_Y,ENSG00000281849-PAR_Y,AL732314.6,<retired>,<retired>


In [81]:
gencode_v35_to_v47_genes_hg38_with_names[gencode_v35_to_v47_genes_hg38_with_names["new-ens_gene"] == "ENSG00000281849"]

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name
57634,chrX,386980,405579,.,+,ENSG00000281849.3,ENSG00000281849,AL732314.6,ENSG00000281849,ENSG00000281849
57636,chrX,419157,421980,.,+,ENSG00000280767.3,ENSG00000280767,AL732314.4,ENSG00000281849,ENSG00000281849


In [83]:
gencode_v35_to_v47_genes_hg38_with_names[gencode_v35_to_v47_genes_hg38_with_names["new-ens_gene"] == "ENSG00000292349"]

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name


In [84]:
to_save = GENCODE / 'gencode_v35-to-v47-genes-hg38-with_names-all_info.tsv'
gencode_v35_to_v47_genes_hg38_with_names.to_csv(to_save, sep="\t", index=False)